# GMI-ATC: Final Product

The exploration is complete for this release. The full research notebook and experiments are preserved in `explore/`. This notebook evaluates the final product using standalone `code/` modules.

Final strategy: **GMI-ATC** (Global Momentum Adaptive Defense and Growth with 20-Day Trend Confirmation).
GMI-ATC keeps every GMI-ADG rule (Top10 12-1 momentum, EWMA two-level risk-budget core, 0%-30% offensive sleeve with breadth, volatility-shock, VIX and drawdown gates, 15-session checks, next-close execution, half-speed rebalancing) and adds one offensive-eligibility rule: on each 15-session signal date an offensive asset must also close above its 20-day moving average. Core weights and the offensive share are unchanged by this rule.

Comparisons: **GMI-ATC, GMI-ADG, GMA-EW, VT and STI (ES3.SI)**. GMI-ADG is the same strategy without the 20-day confirmation (ablation). Currency: SGD. Fixed one-way fee: **0.05%**.

Note: the 20-day window was selected on the 2015–2026 research period under both cost scenarios. The rule was introduced on 2026-10-21, so the October 2026 test is reported as an observation rather than independent out-of-sample evidence.

### 1. Evaluation inputs
Change only the investment and interval below. Replace `data/raw/raw sora.csv` with an updated unedited MAS export when needed. For a test period that is not finished, set `ALLOW_PARTIAL=True` to view interim observations.

In [ ]:
INITIAL_CAPITAL_SGD = 10000.0
TEST_START = '2026-10-01'
TEST_END = '2026-10-31'
ALLOW_PARTIAL = True
REFRESH_MARKET_DATA = False


### 2. Raw SORA preparation

The parser removes MAS metadata/footer rows, carries grouped year/month labels forward, and validates value dates, publication dates and annual percentage rates. Processed data and an audit are saved in `data/processed/`. Backtests select the latest published prior-value-date rate and use a simple ACT/365 daily cash proxy.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'code' / 'main.py').is_file() and (p / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the downloaded project folder or one of its subfolders.')
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
from data import process_sora
from product_evaluation import evaluate_final_product
sora_observations = process_sora()
display(sora_observations.tail())


### 3. Evaluate the final product

Historical holdings, pending orders and risk states are replayed before the selected interval. All five series use the same dates and fee rate. PENDING means the requested period has not finished; PARTIAL identifies an explicitly requested interim observation. Historical validation inspected during research must not be described as an untouched holdout.

Unfilled observed quotes are checked against each exchange calendar. Trailing quote delays shorten explicitly requested partial results to a common complete date; interior quote gaps fail. Read `observed_quote_coverage.csv` and any partial-result explanation. Calendar coverage must include the evaluation year; update requirements if a future holiday calendar is unavailable.

In [ ]:
report = evaluate_final_product(
    start=TEST_START, end=TEST_END, initial_capital=INITIAL_CAPITAL_SGD,
    allow_partial=ALLOW_PARTIAL, refresh=REFRESH_MARKET_DATA, output_dir=PROJECT_ROOT / 'output')
status = report['status'] if isinstance(report.get('status'), dict) else report
print({key: status[key] for key in ['strategy', 'status', 'requested_start', 'requested_end', 'actual_start', 'actual_end', 'observations', 'initial_capital_sgd'] if key in status})
if status.get('reason') or status.get('partial_reason'):
    print(status.get('reason') or status.get('partial_reason'))
if 'metrics' in report:
    percentage_metrics = {'Cumulative Return', 'Annualized Return', 'Annualized Volatility',
        'Maximum Drawdown', 'Positive-Day Ratio', 'Annual One-Way Turnover',
        'Annual Two-Way Traded Notional', 'Average Gross Exposure'}
    display(report['metrics'].style.format({c: '{:.2%}' if c in percentage_metrics else '{:.2f}'
        for c in report['metrics']}, na_rep='N/A'))
    percentage_course = {'Annualized Return', 'Annualized Volatility', 'Maximum Drawdown',
        'Trade Win Rate', 'Average FIFO Match Return'}
    display(report['course_metrics'].style.format({c: '{:.2%}' if c in percentage_course else '{:.2f}'
        for c in report['course_metrics']}, na_rep='N/A'))


### 4. Figures and submission outputs

Figures show cumulative net returns, wealth, filled drawdowns and actual asset-class capital weights. Tables and the 18-asset daily closing-price exports are in `output/tables/`. Source CSVs retain full calculation precision. Check `output/evaluation_status.json` for the actual dates of the latest run.

In [ ]:
from IPython.display import Image, display
for image_path in report.get('charts', {}).values():
    display(Image(filename=image_path))
